# <center>Intelligent Resume Optimization System Using NLP and Generative AI </center>


In [1]:
#Install Libraries
import subprocess, sys
libraries = ['pypdf', 'python-docx', 'scikit-learn', 'requests', 'sentence-transformers', 'keybert']
for lib in libraries:
    print(f'Installing {lib}...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', lib, '-q'], check=False)
print('Done!')

Installing pypdf...
Installing python-docx...
Installing scikit-learn...
Installing requests...
Installing sentence-transformers...
Installing keybert...
Done!


In [2]:
# Import Libraries
import io, re, os, json, requests
from collections import Counter
from sentence_transformers import SentenceTransformer
from keybert import KeyBERT

try:
    from pypdf import PdfReader
    print('pypdf ready')
except:
    PdfReader = None
    print('pypdf not found')

try:
    from docx import Document
    print('python-docx ready')
except:
    Document = None
    print('python-docx not found')

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    print('scikit-learn ready')
except:
    TfidfVectorizer = None
    cosine_similarity = None
    print('scikit-learn not found')

print('All imports done!')

pypdf ready
python-docx ready
scikit-learn ready
All imports done!


In [61]:
# All Helper Functions

STOPWORDS = {
    'a','an','and','are','as','at','be','by','for','from','has',
    'have','in','is','it','of','on','or','that','the','this','to',
    'with','you','your','we','our','will','can','using','use','into',
    'within','across','their','they','job','role','candidate','work',
    'experience','skills','team','teams','responsibilities','required',
}

SYNONYMS = {
    'developed':'built', 'created':'built', 'designed':'built',
    'implemented':'built', 'programmed':'built', 'coded':'built',
    'managed':'led', 'supervised':'led', 'handled':'led',
    'utilized':'used', 'leveraged':'used', 'employed':'used',
    'constructed':'built', 'established':'built',
    'ml':'machine learning', 'ai':'artificial intelligence',
    'nlp':'natural language processing', 'dl':'deep learning',
    'cv':'computer vision', 'db':'database',
    'analysed':'analyzed', 'modelling':'modeling',
    'analyse':'analyze', 'optimisation':'optimization',
}

# Fixed list of real technical skills to scan for in JD
TECH_SKILL_LIST = [
    'python','sql','r','pandas','numpy','scikit-learn','tensorflow',
    'pytorch','xgboost','random forest','decision tree','matplotlib',
    'seaborn','plotly','power bi','tableau','excel','apache spark',
    'hadoop','nlp','computer vision','machine learning','deep learning',
    'statistical modeling','hypothesis testing','regression analysis',
    'time series','feature engineering','data wrangling','smote',
    'hyperparameter optimization','data visualization','jupyter',
    'github','clustering','classification','regression','forecasting',
    'data analytics','data science','predictive modeling','spss',
    'big data','business intelligence','data cleaning','data preprocessing',
]

def clean_text(text):
    if not text: return ''
    text = re.sub(r'\r', '\n', text)
    text = re.sub(r'\b([A-Za-z])\s([A-Za-z])\s([A-Za-z])\s([A-Za-z])\b', r'\1\2\3\4', text)
    text = re.sub(r'\b([A-Za-z])\s([A-Za-z])\s([A-Za-z])\b', r'\1\2\3', text)
    text = re.sub(r'([a-z])\s([a-z]{1,2})\s([a-z])', r'\1\2\3', text)
    text = re.sub(r'(\w)-\s+(\w)', r'\1\2', text)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def normalize(text):
    text = text.lower()
    for word, replacement in SYNONYMS.items():
        text = re.sub(r'\b' + word + r'\b', replacement, text)
    return text

def read_pdf(file_path):
    if PdfReader is None: return ''
    try:
        reader = PdfReader(file_path)
        pages = [page.extract_text() or '' for page in reader.pages]
        raw = '\n'.join(pages)
        cleaned = clean_text(raw)
        words = cleaned.split()
        if words:
            single_chars = sum(1 for w in words if len(w) == 1)
            if single_chars / len(words) > 0.3:
                cleaned = re.sub(r'(?<![.\n])\b([A-Za-z])\b\s', r'\1', cleaned)
                cleaned = clean_text(cleaned)
        return cleaned
    except Exception as e:
        print(f'PDF read error: {e}')
        return ''

def read_docx(file_path):
    if Document is None: return ''
    doc = Document(file_path)
    parts = []
    for para in doc.paragraphs:
        if para.text.strip(): parts.append(para.text.strip())
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                for para in cell.paragraphs:
                    if para.text.strip(): parts.append(para.text.strip())
    return clean_text('\n'.join(parts))

def read_file(file_path):
    file_path = str(file_path)
    if file_path.endswith('.pdf'):    return read_pdf(file_path)
    elif file_path.endswith('.docx'): return read_docx(file_path)
    elif file_path.endswith('.txt'):
        with open(file_path, encoding='utf-8', errors='ignore') as f:
            return clean_text(f.read())
    return ''

def tokenize(text):
    words = re.findall(r'[a-zA-Z][a-zA-Z+#.\-]{1,}', text.lower())
    return [w.strip('.-') for w in words if w not in STOPWORDS and len(w) > 2]

def technical_skills_from_jd(jd_text):
    """Return only real technical skills found in JD."""
    jd_lower = jd_text.lower()
    return [skill for skill in TECH_SKILL_LIST if skill in jd_lower]

def missing_keywords(resume_text, jd_text):
    """Find real technical skills in JD that are missing from resume."""
    jd_skills    = technical_skills_from_jd(jd_text)
    resume_lower = normalize(resume_text).lower()
    return [s for s in jd_skills if s not in resume_lower][:12]

# ── Scoring Method 1: TF-IDF ─────────────────────────────
def tfidf_score(resume_text, jd_text):
    if not TfidfVectorizer: return 0.0
    vectorizer = TfidfVectorizer(
        stop_words='english', ngram_range=(1,2),
        max_features=1000, sublinear_tf=True,
    )
    matrix = vectorizer.fit_transform([normalize(resume_text), normalize(jd_text)])
    return round(float(cosine_similarity(matrix[0], matrix[1])[0][0]) * 100, 1)

# ── Scoring Method 2: Keyword Overlap ────────────────────
def keyword_overlap_score(resume_text, jd_text):
    resume_words = set(tokenize(normalize(resume_text)))
    jd_words     = set(tokenize(normalize(jd_text)))
    if not jd_words: return 0.0
    return round(len(resume_words & jd_words) / len(jd_words) * 100, 1)

# ── Scoring Method 3: Technical Skill Match ──────────────
def weighted_keyword_score(resume_text, jd_text):
    jd_skills    = technical_skills_from_jd(jd_text)
    resume_lower = normalize(resume_text).lower()
    if not jd_skills: return 0.0
    matched = sum(1 for s in jd_skills if s in resume_lower)
    return round(matched / len(jd_skills) * 100, 1)

# ── Scoring Method 4: Semantic (BERT) ────────────────────
semantic_model = SentenceTransformer('all-MiniLM-L6-v2')

def semantic_score(resume_text, jd_text):
    emb1 = semantic_model.encode([resume_text])
    emb2 = semantic_model.encode([jd_text])
    return float(round(cosine_similarity(emb1, emb2)[0][0] * 100, 1))

# ── Combined Score — balanced weights ────────────────────
def calculate_match_score(resume_text, jd_text):
    if not resume_text or not jd_text: return 0.0
    s1 = tfidf_score(resume_text, jd_text)           # 15%
    s2 = keyword_overlap_score(resume_text, jd_text)  # 35%
    s3 = weighted_keyword_score(resume_text, jd_text) # 35%
    s4 = semantic_score(resume_text, jd_text)         # 15%
    return round((s1 * 0.15) + (s2 * 0.35) + (s3 * 0.35) + (s4 * 0.15), 1)

print('All functions loaded!')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

All functions loaded!


In [62]:
 #Load Resume and JD

RESUME_FILE = 'PRAISY UTHUPPAN-Resume.docx'        
JD_FILE     = 'job_description.txt'

resume_text = read_file(RESUME_FILE) if os.path.exists(RESUME_FILE) else ''
jd_text     = read_file(JD_FILE)     if os.path.exists(JD_FILE)     else ''

if not resume_text.strip():
    print(f'ERROR: Could not find {RESUME_FILE}')
    print(f'Current folder: {os.getcwd()}')
else:
    print(f'Resume loaded : {len(resume_text)} characters')

if not jd_text.strip():
    print(f'ERROR: Could not find {JD_FILE}')
else:
    print(f'JD loaded     : {len(jd_text)} characters')

print()
print('=== RESUME TEXT PREVIEW ===')
print(resume_text[:400])
print()

words = resume_text.split()
if words:
    single_chars = sum(1 for w in words if len(w) == 1)
    ratio = single_chars / len(words)
    if ratio > 0.2:
        print(f'WARNING: {round(ratio*100)}% single chars — PDF may be broken')
        print('Consider using DOCX version for better accuracy')
    else:
        print(f'PDF quality: GOOD ({round(ratio*100)}% single chars)')

Resume loaded : 6419 characters
JD loaded     : 2944 characters

=== RESUME TEXT PREVIEW ===
PRAISY UTHUPPAN
DATA SCIENTIST
praisyuthuppan@gmail.com 7594800978 MALAPPURAM
LinkedIn GitHub
Professional Summary
Entry-Level Data Scientist with an M.Sc. in Statistics, IBM Data Science Professional Certificate, and practical experienceinmachine learning, statistical modelling, and data analytics. Built predictive models withupto 93% accuracy, analysed 9,500+ restaurant records, and developed da

PDF quality: GOOD (5% single chars)


In [63]:
#  ATS Score Before Rewrite

s1    = tfidf_score(resume_text, jd_text)
s2    = keyword_overlap_score(resume_text, jd_text)
s3    = weighted_keyword_score(resume_text, jd_text)
s4    = semantic_score(resume_text, jd_text)
score = calculate_match_score(resume_text, jd_text)
gaps  = missing_keywords(resume_text, jd_text)
jd_skills = technical_skills_from_jd(jd_text)

print('=' * 55)
print(f'  FINAL ATS MATCH SCORE : {score}%')
print('=' * 55)
print(f'  TF-IDF similarity      : {s1}%   (25% weight)')
print(f'  Keyword overlap        : {s2}%   (25% weight)')
print(f'  Technical skill match  : {s3}%   (25% weight)')
print(f'  Semantic similarity    : {s4}%   (25% weight)')
print('=' * 55)

filled = int(score / 5)
bar = '█' * filled + '░' * (20 - filled)
print(f'\n  [{bar}] {score}%')

if score >= 75:
    print('  Status: GOOD — Strong match')
elif score >= 55:
    print('  Status: AVERAGE — Needs improvement')
elif score >= 35:
    print('  Status: WEAK — Needs significant tailoring')
else:
    print('  Status: VERY LOW — Major tailoring needed')

print()
print('JD technical skills found :', jd_skills)
print()
print('Missing from your resume  :', gaps)

  FINAL ATS MATCH SCORE : 70.4%
  TF-IDF similarity      : 30.9%   (25% weight)
  Keyword overlap        : 56.7%   (25% weight)
  Technical skill match  : 94.7%   (25% weight)
  Semantic similarity    : 85.4000015258789%   (25% weight)

  [██████████████░░░░░░] 70.4%
  Status: AVERAGE — Needs improvement

JD technical skills found : ['python', 'sql', 'r', 'pandas', 'numpy', 'scikit-learn', 'tensorflow', 'pytorch', 'xgboost', 'random forest', 'decision tree', 'matplotlib', 'seaborn', 'plotly', 'power bi', 'tableau', 'excel', 'apache spark', 'hadoop', 'nlp', 'computer vision', 'machine learning', 'deep learning', 'hypothesis testing', 'regression analysis', 'time series', 'feature engineering', 'data wrangling', 'smote', 'hyperparameter optimization', 'data visualization', 'clustering', 'classification', 'regression', 'data analytics', 'data science', 'big data', 'business intelligence']

Missing from your resume  : ['nlp', 'deep learning']


In [64]:
# Groq API Key

GROQ_API_KEY = "Your_groq_api_key"   

try:
    response = requests.post(
        'https://api.groq.com/openai/v1/chat/completions',
        headers={
            'Authorization': f'Bearer {GROQ_API_KEY}',
            'Content-Type': 'application/json',
        },
        json={
            'model': 'llama-3.3-70b-versatile',
            'messages': [{'role': 'user', 'content': 'say hello'}],
            'max_tokens': 10,
        },
        timeout=15,
    )
    data = response.json()
    if 'choices' in data:
        print('Groq API is working!')
        print('Model : llama-3.3-70b-versatile')
    else:
        print('Error:', data)
except Exception as e:
    print('Failed:', str(e))

Groq API is working!
Model : llama-3.3-70b-versatile


In [65]:
def call_groq(prompt, max_tokens=4000):
    try:
        response = requests.post(
            'https://api.groq.com/openai/v1/chat/completions',
            headers={
                'Authorization': f'Bearer {GROQ_API_KEY}',
                'Content-Type': 'application/json',
            },
            json={
                'model': 'llama-3.3-70b-versatile',
                'messages': [{'role': 'user', 'content': prompt}],
                'max_tokens': max_tokens,
                'temperature': 0.2,
            },
            timeout=90,
        )
        data = response.json()
        if 'choices' in data:
            return data['choices'][0]['message']['content'].strip()
        elif 'error' in data:
            return f"Groq Error: {data['error']['message']}"
    except Exception as e:
        return f'Request failed: {str(e)}'


def inject_missing_keywords(rewritten, jd_text):
    still_missing = missing_keywords(rewritten, jd_text)
    if not still_missing:
        return rewritten
    lines = rewritten.splitlines()
    new_lines = []
    skills_found = False
    for line in lines:
        new_lines.append(line)
        if re.search(r'^(SKILLS|TECHNICAL SKILLS|CORE COMPETENCIES)$',
                     line.strip(), re.IGNORECASE) and not skills_found:
            skills_found = True
            new_lines.append(', '.join(s.title() for s in still_missing[:10]))
    if not skills_found and still_missing:
        final = []
        inserted = False
        for line in new_lines:
            final.append(line)
            if not inserted and len(line.strip()) > 50:
                final.append('Additional skills: ' + ', '.join(s.title() for s in still_missing[:8]) + '.')
                inserted = True
        return clean_text('\n'.join(final))
    return clean_text('\n'.join(new_lines))


def rewrite_resume(resume_text, jd_text):
    gaps     = missing_keywords(resume_text, jd_text)
    skill_kw = technical_skills_from_jd(jd_text)

    prompt = f"""You are a senior professional resume writer and ATS specialist.

Rewrite the resume to professionally match the job description.

TECHNICAL SKILLS TO INCLUDE: {', '.join(skill_kw)}
MISSING SKILLS TO ADD NATURALLY: {', '.join(gaps[:10])}

STRICT RULES:
1. READ THE ENTIRE RESUME before writing anything.
2. Every section in the original resume MUST appear in the rewrite.
3. Do NOT write "No direct experience" — the experience IS in the resume.
4. Do NOT invent fake companies, degrees, dates, or numbers.
5. Keep all real data: name, email, phone, LinkedIn, dates, GPA.
6. Rewrite summary to match the JD role language closely.
7. Skills section must list ALL technical skills the candidate has.
8. Rewrite every bullet point with strong action verbs and JD keywords.
9. Write every project FULLY — tools used, dataset size, accuracy achieved.
10. No markdown bold (**text**) — plain text only.
11. Section headings in plain ALL CAPS — no symbols or asterisks.
12. Do NOT add "Analytics Hub", "Work Big Data" or any recruitment phrases.

FORMAT TO FOLLOW EXACTLY:
CANDIDATE NAME
email | phone | city | linkedin

SUMMARY
2-3 sentences matching the JD.

SKILLS
Programming: Python, R, SQL
Machine Learning: Scikit-learn, TensorFlow, PyTorch, XGBoost, Random Forest
Data Visualization: Power BI, Tableau, Matplotlib, Seaborn, Plotly
Statistics: Regression, Hypothesis Testing, Time Series, Clustering
Big Data: Apache Spark, Hadoop
Tools: Excel, Jupyter Notebook, GitHub, SPSS

INTERNSHIP
Company | Role | Date
- Action verb + task + tool + result
- Action verb + task + tool + result

PROJECTS
Project Title | Tools used
- Full description, dataset, accuracy/result achieved
- Techniques used and business insight

EDUCATION
Degree | Institution | Year | CGPA

CERTIFICATIONS
Name | Issuer | Year

--- ORIGINAL RESUME ---
{resume_text}

--- JOB DESCRIPTION ---
{jd_text}

Write the complete professional resume now:"""

    rewritten = call_groq(prompt, max_tokens=4000)

    if rewritten and not rewritten.startswith('Groq Error'):
        rewritten = inject_missing_keywords(rewritten, jd_text)

    return rewritten

print('Groq AI functions ready!')

Groq AI functions ready!


In [66]:
# CELL 8 — Run the AI Rewrite

print('Sending to Groq AI (llama-3.3-70b)...')
print('Please wait 10-20 seconds...\n')

rewritten_resume = rewrite_resume(resume_text, jd_text)

if rewritten_resume.startswith('Groq Error') or rewritten_resume.startswith('Request failed'):
    print('ERROR:', rewritten_resume)
else:
    s1_a = tfidf_score(rewritten_resume, jd_text)
    s2_a = keyword_overlap_score(rewritten_resume, jd_text)
    s3_a = weighted_keyword_score(rewritten_resume, jd_text)
    s4_a = semantic_score(rewritten_resume, jd_text)
    score_after = calculate_match_score(rewritten_resume, jd_text)

    print('=' * 55)
    print('  REWRITE COMPLETE!')
    print('=' * 55)
    print(f'  Score BEFORE : {score}%')
    print(f'  Score AFTER  : {score_after}%')
    print(f'  Improvement  : +{round(score_after - score, 1)}%')
    print('=' * 55)
    print(f'  TF-IDF      : {s1}% → {s1_a}%')
    print(f'  KW Overlap  : {s2}% → {s2_a}%')
    print(f'  Tech Skills : {s3}% → {s3_a}%')
    print(f'  Semantic    : {s4}% → {s4_a}%')
    print('=' * 55)
    print()
    print(rewritten_resume)

Sending to Groq AI (llama-3.3-70b)...
Please wait 10-20 seconds...

  REWRITE COMPLETE!
  Score BEFORE : 70.4%
  Score AFTER  : 71.0%
  Improvement  : +0.6%
  TF-IDF      : 30.9% → 29.6%
  KW Overlap  : 56.7% → 55.4%
  Tech Skills : 94.7% → 97.4%
  Semantic    : 85.4000015258789% → 86.9000015258789%

PRAISY UTHUPPAN
praisyuthuppan@gmail.com | 7594800978 | MALAPPURAM | LinkedIn

SUMMARY
Asadetail-driven and analytical data scientist, I leveragemystrong foundationinmachine learning, statistical modelling, and data analyticstodrive business growth and informed decision-making. With hands-on experienceinbuilding predictive models, data visualization, and big data technologies, I am excitedtoapplymyskillsina junior data scientist roletodeliver data-driven solutions and insights. My expertise in Python, SQL, R, and data science tools enablesmeto transform complex datasets into actionable business insights and drive customer-centric strategies.

SKILLS
Nlp, Deep Learning, Business Intelligenc

In [67]:
# DEBUG CELL — run this to see exact breakdown
print('=== SCORE BREAKDOWN DIAGNOSIS ===')
print()

s1 = tfidf_score(resume_text, jd_text)
s2 = keyword_overlap_score(resume_text, jd_text)
s3 = weighted_keyword_score(resume_text, jd_text)
s4 = semantic_score(resume_text, jd_text)

print(f'TF-IDF score        : {s1}%')
print(f'Keyword overlap     : {s2}%')
print(f'Tech skill match    : {s3}%')
print(f'Semantic score      : {s4}%')
print()

# Show which tech skills are found vs missing
jd_skills    = technical_skills_from_jd(jd_text)
resume_lower = normalize(resume_text).lower()

print('JD skills found:')
for s in jd_skills:
    found = '✓' if s in resume_lower else '✗'
    print(f'  {found} {s}')

print()
print('Total JD skills   :', len(jd_skills))
print('Matched in resume :', sum(1 for s in jd_skills if s in resume_lower))
print()
print('Resume length     :', len(resume_text), 'chars')
print('Resume preview    :')
print(resume_text[:300])

=== SCORE BREAKDOWN DIAGNOSIS ===

TF-IDF score        : 30.9%
Keyword overlap     : 56.7%
Tech skill match    : 94.7%
Semantic score      : 85.4000015258789%

JD skills found:
  ✓ python
  ✓ sql
  ✓ r
  ✓ pandas
  ✓ numpy
  ✓ scikit-learn
  ✓ tensorflow
  ✓ pytorch
  ✓ xgboost
  ✓ random forest
  ✓ decision tree
  ✓ matplotlib
  ✓ seaborn
  ✓ plotly
  ✓ power bi
  ✓ tableau
  ✓ excel
  ✓ apache spark
  ✓ hadoop
  ✗ nlp
  ✓ computer vision
  ✓ machine learning
  ✗ deep learning
  ✓ hypothesis testing
  ✓ regression analysis
  ✓ time series
  ✓ feature engineering
  ✓ data wrangling
  ✓ smote
  ✓ hyperparameter optimization
  ✓ data visualization
  ✓ clustering
  ✓ classification
  ✓ regression
  ✓ data analytics
  ✓ data science
  ✓ big data
  ✓ business intelligence

Total JD skills   : 38
Matched in resume : 36

Resume length     : 6419 chars
Resume preview    :
PRAISY UTHUPPAN
DATA SCIENTIST
praisyuthuppan@gmail.com 7594800978 MALAPPURAM
LinkedIn GitHub
Professional Summary
Entry-Le

In [68]:
# CELL 9 — Save as DOCX and TXT

def save_as_docx(text, output_path='tailored_resume.docx'):
    if Document is None:
        print('Run: pip install python-docx')
        return
    doc = Document()
    doc.add_heading('Tailored Resume', level=1)
    for block in text.split('\n\n'):
        block = block.strip()
        if not block: continue
        lines = block.splitlines()
        if len(lines) == 1 and lines[0].isupper():
            doc.add_heading(lines[0].title(), level=2)
            continue
        for line in lines:
            line = line.strip()
            if not line: continue
            if line.isupper():
                doc.add_heading(line.title(), level=2)
            elif line.startswith('- ') or line.startswith('• '):
                doc.add_paragraph(line[2:], style='List Bullet')
            else:
                doc.add_paragraph(line)
    doc.save(output_path)
    print(f'Saved DOCX: {os.path.abspath(output_path)}')

save_as_docx(rewritten_resume)

with open('tailored_resume.txt', 'w', encoding='utf-8') as f:
    f.write(rewritten_resume)
print('Saved TXT : tailored_resume.txt')

Saved DOCX: C:\Users\prais\Documents\Codex\resume rewriter\tailored_resume.docx
Saved TXT : tailored_resume.txt


In [69]:
# CELL 10 — Resume Coach Chatbot
# Change question and re-run anytime

def ask_resume_coach(question):
    current_score = calculate_match_score(resume_text, jd_text)
    gaps = missing_keywords(resume_text, jd_text)
    prompt = f"""You are a professional resume coach.
Match score: {current_score}%
Missing skills: {', '.join(gaps) or 'none'}
Resume: {resume_text[:1800]}
Job description: {jd_text[:1200]}
Question: {question}
Give short practical advice. Do not invent experience."""
    return call_groq(prompt, max_tokens=600)

my_question = 'What are the top 3 things I should improve in my resume for this job?'
print(f'Q: {my_question}')
print('-' * 55)
print(ask_resume_coach(my_question))

Q: What are the top 3 things I should improve in my resume for this job?
-------------------------------------------------------
Based on the job description and your resume, here are the top 3 things you should improve:

1. **Emphasize NLP skills**: Although you've mentioned NLP in your skills section, you haven't highlighted any experience or projects related to it. If you have any relevant coursework or personal projects, mention them to demonstrate your NLP skills.
2. **Highlight deep learning experience**: Similar to NLP, you've listed PyTorch and TensorFlow, but it's not clear if you have hands-on experience with deep learning. If you have any relevant projects or experience, highlight them to showcase your skills.
3. **Quantify your achievements**: While you've mentioned building predictive models with up to 93% accuracy, try to be more specific about the impact of your work. For example, you could mention "Improved customer churn prediction accuracy by 25% using machine learnin